# Encore: first KPI, repertoire age

How old is the material a band plays live? The **repertoire age** of a
performance is the show's year minus the release year of the song (its
earliest studio album, or its earliest recording when it is not on an
album). A song played before its release year counts as age 0.

This notebook reads **only the aggregated marts in the `analytics` schema**
(`mart_repertoire_age`, `mart_match_quality`). It never touches raw data, and
it holds no setlists, dates, venues or song titles.

Run the pipeline first (`docker compose -f infra/docker-compose.yml --env-file .env up -d`, then
a full `encore_pipeline` run) so the marts are populated.

Show data: [setlist.fm](https://www.setlist.fm) (aggregated only). Discography: [MusicBrainz](https://musicbrainz.org) (CC0).

## Setup

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import psycopg2
from dotenv import load_dotenv

# Repo root = first parent folder holding CLAUDE.md (works from notebooks/ or the root).
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "CLAUDE.md").exists())
load_dotenv(ROOT / ".env")

# Postgres is published on the host at 127.0.0.1:5435 (docs/context/tech.md).
DB = dict(
    host=os.environ.get("ENCORE_DB_HOST", "localhost"),
    port=int(os.environ.get("ENCORE_DB_PORT", "5435")),
    user=os.environ["POSTGRES_USER"],
    password=os.environ["POSTGRES_PASSWORD"],
    dbname="encore",
)

# The only tables this notebook may read. Anything else is refused.
ANALYTICS_TABLES = {"mart_repertoire_age", "mart_match_quality"}
TEXT_COLUMNS = {"band", "tour_name"}


def read_analytics(table: str) -> pd.DataFrame:
    """Read one analytics mart through a read-only session."""
    if table not in ANALYTICS_TABLES:
        raise ValueError(f"{table!r} is not an allowed analytics table")
    conn = psycopg2.connect(**DB)
    try:
        conn.set_session(readonly=True)
        with conn.cursor() as cur:
            cur.execute(f"select * from analytics.{table}")
            columns = [c.name for c in cur.description]
            df = pd.DataFrame(cur.fetchall(), columns=columns)
    finally:
        conn.close()
    df = df.drop(columns=["computed_at"])
    # Postgres numeric arrives as Decimal; make the numbers plain floats/ints.
    return df.apply(lambda s: s if s.name in TEXT_COLUMNS else pd.to_numeric(s))


# One fixed colour per band (the colour follows the band, never its rank), in
# the project's band order. Light-surface values of the reference categorical palette.
BANDS = [
    "Arctic Monkeys", "Oasis", "Linkin Park", "Twenty One Pilots",
    "Muse", "Metallica", "Avenged Sevenfold",
]
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7"]
BAND_COLOR = dict(zip(BANDS, PALETTE))
INK, INK_MUTED, GRID, NEUTRAL = "#0b0b0b", "#52514e", "#e6e5e1", "#a3a29c"

plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
    "axes.edgecolor": GRID, "axes.labelcolor": INK_MUTED,
    "xtick.color": INK_MUTED, "ytick.color": INK_MUTED, "text.color": INK,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "axes.axisbelow": True, "font.size": 10,
})

age = read_analytics("mart_repertoire_age")
quality = read_analytics("mart_match_quality")
print(f"mart_repertoire_age: {len(age)} rows · mart_match_quality: {len(quality)} rows")
print("bands present:", ", ".join(sorted(age["band"].unique())))

## 1. Average repertoire age by year, one line per band

The mart is split by tour, so a year's average is the tour averages weighted by
each cell's matched performances. A line breaks where a band played no shows
(for example a hiatus) instead of joining the years across the gap.

In [ ]:
scored = age.dropna(subset=["avg_repertoire_age"]).copy()
scored["weighted"] = scored["avg_repertoire_age"] * scored["matched_performances"]
by_year = (
    scored.groupby(["band", "show_year"])
    .agg(weighted=("weighted", "sum"), n=("matched_performances", "sum"))
    .reset_index()
)
by_year["avg_age"] = by_year["weighted"] / by_year["n"]

fig, ax = plt.subplots(figsize=(10, 4.5))
bands_present = [b for b in BANDS if b in set(by_year["band"])]
for band in bands_present:
    series = by_year[by_year["band"] == band].set_index("show_year")["avg_age"]
    # Reindex to every year in range so missing years become gaps in the line.
    series = series.reindex(range(int(series.index.min()), int(series.index.max()) + 1))
    ax.plot(series.index, series.values, color=BAND_COLOR[band], linewidth=2,
            marker="o", markersize=7, markeredgecolor="#fcfcfb", markeredgewidth=1.5,
            label=band)
    last = series.dropna()
    if len(bands_present) <= 4:  # direct label at the last point
        ax.annotate(band, (last.index[-1], last.iloc[-1]), xytext=(8, 0),
                    textcoords="offset points", va="center", color=INK, fontsize=9)

ax.set_title("Average repertoire age by year", loc="left", fontsize=12, color=INK)
ax.set_xlabel("Show year")
ax.set_ylabel("Average age of songs played (years)")
ax.set_ylim(bottom=0)
ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
if len(bands_present) > 1:
    ax.legend(frameon=False, loc="upper left", ncol=min(len(bands_present), 4))
fig.tight_layout()
plt.show()

## 2. Average repertoire age by tour, one band

Set `BAND` to any band present in the data. A tour that spans several years is
one bar (weighted by matched performances) and its label shows the years.
Shows with no tour name on setlist.fm are pooled into **Unknown tour** (grey,
last): a bucket of many unrelated shows, not a real tour.

In [ ]:
BAND = "Oasis"

band_age = age[(age["band"] == BAND)].dropna(subset=["avg_repertoire_age"]).copy()
band_age["weighted"] = band_age["avg_repertoire_age"] * band_age["matched_performances"]
tours = (
    band_age.groupby("tour_name")
    .agg(weighted=("weighted", "sum"), n=("matched_performances", "sum"),
         first_year=("show_year", "min"), last_year=("show_year", "max"))
    .reset_index()
)
tours["avg_age"] = tours["weighted"] / tours["n"]
tours["is_unknown"] = tours["tour_name"] == "Unknown tour"
tours = tours.sort_values(["is_unknown", "first_year", "tour_name"]).reset_index(drop=True)
tours["label"] = tours.apply(
    lambda r: r["tour_name"] if r["is_unknown"]
    else f'{r["tour_name"]} ({r["first_year"]}'
         + ("" if r["first_year"] == r["last_year"] else f'\u2013{r["last_year"]}') + ")",
    axis=1,
)

fig, ax = plt.subplots(figsize=(10, 0.42 * len(tours) + 1.6))
colors = [NEUTRAL if u else BAND_COLOR.get(BAND, PALETTE[0]) for u in tours["is_unknown"]]
ax.barh(tours["label"], tours["avg_age"], height=0.6, color=colors)
for y, value in enumerate(tours["avg_age"]):
    ax.text(value + 0.08, y, f"{value:.1f}", va="center", fontsize=9, color=INK_MUTED)
ax.invert_yaxis()  # chronological, top to bottom
ax.set_title(f"{BAND}: average repertoire age by tour", loc="left", fontsize=12, color=INK)
ax.set_xlabel("Average age of songs played (years)")
ax.grid(axis="y", visible=False)
ax.set_xlim(left=0)
fig.tight_layout()
plt.show()

## 3. Match quality by band and year

How well setlist songs resolve to the song catalog. The headline metric is
**matched by performance**: the share of performances (weighted by plays) that
matched a catalog song. **Matched by title** counts each distinct song once, so
it is the harsher view and drops first if matching degrades.

In [ ]:
totals = (
    quality.groupby("band")
    .agg(performances=("performances", "sum"), matched=("matched_performances", "sum"))
    .reset_index()
)
totals["matched by performance"] = totals["matched"] / totals["performances"]
display(
    totals.rename(columns={"matched": "matched performances"})
    .style.format({"performances": "{:,}", "matched performances": "{:,}",
                   "matched by performance": "{:.2%}"})
    .hide(axis="index")
)

table = quality.sort_values(["band", "show_year"]).rename(columns={
    "band": "band", "show_year": "year", "performances": "performances",
    "matched_performances": "matched performances",
    "match_rate_by_performance": "matched by performance",
    "distinct_songs": "distinct songs", "distinct_songs_matched": "distinct songs matched",
    "match_rate_by_title": "matched by title",
})
display(
    table.style.format({
        "year": "{:d}", "performances": "{:,}", "matched performances": "{:,}",
        "matched by performance": "{:.2%}", "matched by title": "{:.2%}",
    }).hide(axis="index")
)